In [1]:
# %load_ext autoreload
# %autoreload 2

In [2]:
from moseplib.data import pointcloud_processing, timeseries_processing, pc_statistics
from moseplib.data.utils import flatten_multiindex, normalize_df
from mosep_analysis.data.config import (
    TARGET_EXTENTS_VIF,
    TARGET_EXTENTS_VIF_SPLITS,
    INTERIM_DATA_FOLDER,
    PROCESSED_DATA_FOLDER,
)
from mosep_analysis.visualization.timeseries import (
    plot_timeseries_separate_axes,
    target_stat_vs_precipitation,
    histograms_targets_attime,
    histogram_targets_interactive,
)

import dask.config
import pandas as pd
import panel as pn
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pointcloudset import Dataset

pn.extension("plotly")
dask.config.set(temporary_directory="/datalocal/chg/tmp/dask")

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## Load Met Data from Bagfile

In [3]:
BAG_NAME: str = "molisens_met_2023_08_07-15_36_45_converted"
DATA_DIR: str = str(INTERIM_DATA_FOLDER / "ViF_Roof" / "data")
TIMESTAMPS_HISTOGRAMS: list = ["2023-08-07T13:37:00"]
SHIFT_WS_DATA = 0  # seconds

In [4]:
# Parameters
BAG_NAME = "molisens_met_2023_08_29-06_04_46_converted"
DATA_DIR = "/datalocal/chg/MOLISENSext/ViF_Roof/data"
TIMESTAMPS_HISTOGRAMS = ["2023-08-29T04:05:00", "2023-08-29T04:25:00", "2023-08-29T04:35:00", "2023-08-29T04:42:00", "2023-08-29T04:50:00"]


In [5]:
df = timeseries_processing.load_timeseries(
    DATA_DIR,
    BAG_NAME,
    topics=("/sensing/aws/ws100_measurements", "/sensing/aws/ws501_measurements"),
    safe_parquet=PROCESSED_DATA_FOLDER / f"weather_df_{BAG_NAME}.parquet",
    verbose=True,
)
if SHIFT_WS_DATA != 0:
    df = df.shift(periods=SHIFT_WS_DATA, freq="s")

Searching for pointcloudset files in:
/datalocal/chg/MOLISENSext/processed/weather_df_molisens_met_2023_08_29-06_04_46_converted.parquet

Found parquet files, loading timeseries data...

### Met Data Overview

In [6]:
parameters = {
    "precipitation intensity": {
        "data_key": ("precipitation", "intensity_hour_shifted"),
        "color": "black",
        "axis_title": "Precipitation Intensity (mm/h)",
    },
    "wind speed": {
        "data_key": ("wind", "speed_avg"),
        "color": "blue",
        "axis_title": "Wind Speed (m/s)",
    },
    "temperature": {
        "data_key": ("temperature", "average"),
        "color": "red",
        "axis_title": "Temperature (°C)",
    },
    "humidity": {
        "data_key": ("humidity", "relative_avg"),
        "color": "green",
        "axis_title": "Humidity (%)",
    },
    "pressure": {
        "data_key": ("pressure", "relative_avg"),
        "color": "purple",
        "axis_title": "Pressure (hPa)",
    },
    "radiation": {
        "data_key": ("radiation", "current"),
        "color": "orange",
        "axis_title": "Radiation (W/m²)",
    },
}

plot_timeseries_separate_axes(df, parameters)

### Find the most relevant precipitation parameters for further analysis:

In [7]:
df.precipitation.loc[:, ["absolute", "differential", "total_precipitation_particles", "total_drops"]].corrwith(
    df.precipitation.intensity_hour_shifted
)

Parameter
absolute                         0.152192
differential                     0.515560
total_precipitation_particles    0.275460
total_drops                      0.361068
dtype: float64

In [8]:
fig1 = px.line(df.precipitation.absolute)
fig1.update_traces(line=dict(color="blue"), connectgaps=True)
fig2 = px.line(df.precipitation.differential.resample("10s").sum() * 60)
fig2.update_traces(line=dict(color="red"))
fig3 = px.line(df.precipitation.intensity_hour_shifted)
fig3.update_traces(line=dict(color="green"), connectgaps=True)
# fig6 = px.line(df.precipitation.intensity_hour)
# fig6.update_traces(line=dict(color="lightgreen"))
fig4 = px.line(df.precipitation.total_precipitation_particles.resample("10s").sum())
fig4.update_traces(line=dict(color="gold"))
fig5 = px.line(df.precipitation.total_drops.resample("10s").sum())
fig5.update_traces(line=dict(color="black"))
# fig6 = px.line(df.

# Create a subplot with secondary y-axis
fig = make_subplots(specs=[[{"secondary_y": True}]])

# Add the traces from fig1, fig3, fig4, and fig5 to the primary y-axis
for trace in fig1.data + fig3.data + fig4.data + fig5.data:
    fig.add_trace(trace, secondary_y=False)

# Add the traces from fig2 to the secondary y-axis
for trace in fig2.data:
    fig.add_trace(trace, secondary_y=True)

# Update the layout
fig.update_layout(width=1400, height=600)

fig.show()

From the correlations with `intensity_hour_shifted` and the plot above the following precipitation parameters seem are chosen for further analysis.

In [9]:
RESAMPLE_FREQ = "1min"
columns_and_aggregation = {
    ("precipitation", "intensity_hour_shifted"): "first",
    ("precipitation", "intensity_hour"): "first",
    ("precipitation", "differential"): "sum",
    ("precipitation", "total_precipitation_particles"): "sum",
    ("precipitation", "total_drops"): "sum",
    ("wind", "speed_avg"): "mean",
    ("temperature", "average"): "mean",
    ("humidity", "relative_avg"): "mean",
    ("pressure", "relative_avg"): "mean",
    ("radiation", "current"): "mean",
}

df_relevant = df.loc[:, columns_and_aggregation.keys()]

columns = flatten_multiindex(df_relevant)
df_relevant = df_relevant.resample(RESAMPLE_FREQ).agg({".".join(k): v for k, v in columns_and_aggregation.items()})

In [10]:
# color df by correlation+
df_relevant.corr()
fig = go.Figure(data=go.Heatmap(z=df_relevant.corr(), x=df_relevant.columns, y=df_relevant.columns))
fig.show()

df_relevant.columns = columns_and_aggregation
df_relevant.corr()

precipitation  \
                                            intensity_hour_shifted   
precipitation intensity_hour_shifted                      1.000000   
              intensity_hour                              0.583692   
              differential                                0.985111   
              total_precipitation_particles               0.546141   
              total_drops                                 0.724565   
wind          speed_avg                                  -0.083409   
temperature   average                                    -0.242776   
humidity      relative_avg                                0.216827   
pressure      relative_avg                                0.458070   
radiation     current                                    -0.070813   

                                                                         \
                                            intensity_hour differential   
precipitation intensity_hour_shifted              0.583692     0.985111   
              intensity_hour                      1.000000     0.561996   
              differential                        0.561996     1.000000   
              total_precipitation_particles       0.438225     0.569009   
              total_drops                         0.549518     0.751294   
wind          speed_avg                          -0.040351    -0.097359   
temperature   average                            -0.274212    -0.246420   
humidity      relative_avg                        0.263549     0.216341   
pressure      relative_avg                        0.480903     0.471648   
radiation     current                            -0.029493    -0.080579   

                                                                           \
                                            total_precipitation_particles   
precipitation intensity_hour_shifted                             0.546141   
              intensity_hour                                     0.438225   
              differential                                       0.569009   
              total_precipitation_particles                      1.000000   
              total_drops                                        0.876170   
wind          speed_avg                                          0.364680   
temperature   average                                           -0.448826   
humidity      relative_avg                                       0.283350   
pressure      relative_avg                                       0.761022   
radiation     current                                           -0.112075   

                                                             wind temperature  \
                                            total_drops speed_avg     average   
precipitation intensity_hour_shifted           0.724565 -0.083409   -0.242776   
              intensity_hour                   0.549518 -0.040351   -0.274212   
              differential                     0.751294 -0.097359   -0.246420   
              total_precipitation_particles    0.876170  0.364680   -0.448826   
              total_drops                      1.000000  0.139120   -0.415247   
wind          speed_avg                        0.139120  1.000000   -0.554915   
temperature   average                         -0.415247 -0.554915    1.000000   
humidity      relative_avg                     0.263788  0.394592   -0.929155   
pressure      relative_avg                     0.730089  0.527552   -0.892323   
radiation     current                         -0.168741  0.472219   -0.766447   

                                                humidity     pressure  \
                                            relative_avg relative_avg   
precipitation intensity_hour_shifted            0.216827     0.458070   
              intensity_hour                    0.263549     0.480903   
              differential                      0.216341     0.471648   
              total_precipitation_particles     

## Load PC data

In [11]:
TOPICS = {
    "lid_pts": "/sensing/lidar/points",
    "lid_pts2": "/sensing/lidar/points2",
    "rad_pts": "/sensing/radar/points",
}

In [12]:
# Remove the first 20 seconds of the dataset.
dataset_rain = pointcloud_processing.load_pointcloudset(
    DATA_DIR, BAG_NAME, topic=TOPICS["lid_pts"], verbose=True, invert_axes=["x", "y"], skip_initial_frames=20 * 10
)

Searching for pointcloudset files in:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_29-06_04_46_conver
ted

Dataset loaded from:
/datalocal/chg/MOLISENSext/ViF_Roof/data/pointcloudset/sensing_lidar_points/molisens_met_2023_08_29-06_04_46_conver
ted

 start =    2023-08-29 04:05:19.853657
 end =      2023-08-29 05:04:46.069975
 duration = 0:59:26.216318
 length =   35659
 avg frequency =  10.00 Hz

Inverting axes:
['x', 'y']

### Resample Dataset to 1min

<font color='red'>For now</font>, we will resample the dataset to 1min resolution due to long computation times. Once we have a better understanding of the data, we can adjust the resolution accordingly.

In [13]:
path = PROCESSED_DATA_FOLDER / f"rain_minutes_{BAG_NAME}"
if not path.is_dir():
    ds_minutes = pointcloud_processing.resample_dataset(dataset_rain, "1min", extra_statistics=["std", "sum"])
    # Save the resampled datasets to parquet
    for stat, ds in ds_minutes.items():
        ds.to_file(path / stat, use_orig_filename=False)

# load from parquet
ds_minutes = {}
for p in path.iterdir():
    ds_minutes[p.stem] = Dataset.from_file(p)

## Exploratory Data Analysis (EDA)

### Setup

In [14]:
TIME_AGGREGATION = "mean"
SPACE_AGGREGATION = pc_statistics.mean_intensity
# SPACE_AGGREGATION = pc_statistics.sum_intensity

In [15]:
target_statistics = pointcloud_processing.subset_and_aggregate_dataset(
    dataset=ds_minutes[TIME_AGGREGATION],
    splits=TARGET_EXTENTS_VIF,
    agg_func=SPACE_AGGREGATION,
    return_type="df",
)
target_statistics.columns.name = "target"

In [16]:
subtarget_statistics = pointcloud_processing.subset_and_aggregate_dataset(
    dataset=ds_minutes[TIME_AGGREGATION],
    splits=TARGET_EXTENTS_VIF_SPLITS,
    agg_func=SPACE_AGGREGATION,
    return_type="df",
)
subtarget_statistics.columns.names = ["target", "color"]

#### Normalize Data

In [17]:
NORM_KIND = "standard"

target_statistics_norm = normalize_df(target_statistics, kind=NORM_KIND)
subtarget_statistics_norm = normalize_df(subtarget_statistics, kind=NORM_KIND)
df_relevant_norm = normalize_df(df_relevant, kind=NORM_KIND, fillna=True)

### Mean intensity of full target (all reflectivities)

In [18]:
NORMALIZE = False

if NORMALIZE:
    targ_stats = target_statistics_norm
    weather_stats = df_relevant_norm
else:
    targ_stats = target_statistics
    weather_stats = df_relevant


fig = target_stat_vs_precipitation(
    targ_stats,
    weather_stats.precipitation.intensity_hour_shifted,
    exclude_cols=None,  # ["Target-02"],
    fig_size=(1000, 500),
    precipitation_name="Rainfall Rate",
    yaxis_title="Mean Intensity",
    yaxis2_title="Rainfall Rate [mm/h]",
)
fig.show()
fig = target_stat_vs_precipitation(
    targ_stats,
    weather_stats.wind.speed_avg,
    exclude_cols=None,  # ["Target-02"],
    fig_size=(1000, 500),
    precipitation_name="Wind Speed",
    yaxis_title="Mean Intensity",
    yaxis2_title="Wind Speed [m/s]",
)
fig.show()
fig = target_stat_vs_precipitation(
    targ_stats,
    weather_stats.radiation.current,
    exclude_cols=None,  # ["Target-02"],
    fig_size=(1000, 500),
    precipitation_name="Radiation",
    yaxis_title="Mean Intensity",
    yaxis2_title="Radiation [W/m²]",
)
fig.show()

### Mean intensity of targets spearate by intensity

In [19]:
NORMALIZE = True

if NORMALIZE:
    subtarg_stats = subtarget_statistics_norm
    weather_stats = df_relevant_norm
    title_addon = "Normalized "
else:
    subtarg_stats = subtarget_statistics
    weather_stats = df_relevant
    title_addon = ""

target_stat_vs_precipitation(
    subtarg_stats,
    weather_stats.precipitation.intensity_hour_shifted,
    value_name="intensity",
    precipitation_name="Rainfall Rate",
    yaxis_title=f"{title_addon}Mean Intensities",
    yaxis2_title=f"{title_addon}Rainfall Rate",
    compress_legend=False,
    fig_size=(1200, 600),
)

In [20]:
# timestamp = "2023-08-29T04:37:00"

# ROOF_EXTENT.apply_limits(utils.get_pointcloud_from_timestamp(ds_minutes["mean"], timestamp)).plot(
#     color="intensity",
#     hover_data=["intensity"],
#     overlay={
#         "Target-05": TARGET_EXTENTS_VIF["Target-05"].apply_limits(
#             utils.get_pointcloud_from_timestamp(ds_minutes["mean"], timestamp)
#         )
#     },
# )

### Histograms for each target

https://plotly.com/chart-studio-help/histogram/#normalizing-a-histogram

In [21]:
plot_data = subtarget_statistics.loc[:, pd.IndexSlice[:, "grey"]]
plot_data = plot_data.melt(value_name="intensity")

fig = px.histogram(
    plot_data,
    x="intensity",
    color="target",
    marginal="box",  # can be 'rug', `box`, `violin`
    hover_data=plot_data.columns,
    histfunc="count",  # can be "avg"
    nbins=100,
    histnorm="percent",  # '', 'percent', 'probability', 'density', 'probability density'
    title="Intensity Distribution of Targets (aggregated in space) over time",
    # log_y=True,
)
fig.show()

#### Range

In [22]:
# target_pcs = pointcloud_processing.subset_and_aggregate_dataset(ds_minutes["mean"], TARGET_EXTENTS_VIF_SPLITS)
# app = histogram_targets_interactive(target_pcs)
# app.run_server(jupyter_mode="external")

In [23]:
TARGET = "Target-02"
VALUE = "range"

if TIMESTAMPS_HISTOGRAMS:
    target_pcs = pointcloud_processing.subset_and_aggregate_dataset(ds_minutes["mean"], TARGET_EXTENTS_VIF_SPLITS)
    fig = histograms_targets_attime(
        target_pcs,
        df_relevant_norm,
        TIMESTAMPS_HISTOGRAMS,
        TARGET,
        VALUE,
    )
    fig.show()

#### Intensity

In [24]:
TARGET = "Target-02"
VALUE = "intensity"

if TIMESTAMPS_HISTOGRAMS:
    target_pcs = pointcloud_processing.subset_and_aggregate_dataset(ds_minutes["mean"], TARGET_EXTENTS_VIF_SPLITS)
    fig = histograms_targets_attime(
        target_pcs,
        df_relevant_norm,
        TIMESTAMPS_HISTOGRAMS,
        TARGET,
        VALUE,
    )
    fig.show()

### Correlations: Precipitaiton ~ Intensity

In [25]:
COLOR = "grey"

corr_data = subtarget_statistics_norm.copy()
corr_data.columns = ["_".join(col).strip() for col in corr_data.columns.values]
corr_data = pd.concat([corr_data, df_relevant_norm.precipitation.intensity_hour_shifted], axis=1)
corr_data["n"] = range(len(corr_data))

fig = px.scatter_matrix(
    corr_data,
    dimensions=[
        f"Target-01_{COLOR}",
        f"Target-02_{COLOR}",
        f"Target-03_{COLOR}",
        f"Target-04_{COLOR}",
        f"Target-05_{COLOR}",
        "intensity_hour_shifted",
    ],
    color="n",
)
fig.update_traces(diagonal_visible=False)
fig.update_layout(width=1200, height=1200)
fig.show()

In [26]:
#! TODO Separate the timesereis in before and after minute 42.

In [27]:
correlation = subtarget_statistics_norm.corrwith(df_relevant_norm.precipitation.intensity_hour, method="spearman")

# Print the correlation values
print(correlation)

target     color
Target-01  white   -0.800845
           grey    -0.419514
           black   -0.627937
Target-02  white   -0.853201
           grey    -0.744404
           black   -0.441190
Target-03  white   -0.846364
           grey    -0.844558
           black   -0.827162
Target-04  white   -0.869124
           grey    -0.713863
           black   -0.885472
Target-05  white   -0.371743
           grey    -0.756159
           black    0.298795
dtype: float64


In [28]:
# # TODO This should be applied to the high temporal data at the start of this NB.

# # Cross-correlation is a measure of similarity between two signals, often used to find features or patterns that appear
# # in both. It can be used to determine how much one series is correlated with a lagged version of another series.
# # In the plot, the x-axis represents the lags and the y-axis represents the cross-correlation values. The error bars
# # represent the confidence intervals. If an error bar does not cross the zero line, it suggests that the correlation at
# # that lag is statistically significant.

# from statsmodels.tsa.stattools import ccf

# # Cross-correlation analysis
# ccf_values = ccf(
#     subtarget_statistics_norm["Target-04"]["white"],
#     df_relevant_norm.precipitation.differential,
#     adjusted=True,
#     alpha=0.05,
# )

# x = list(range(len(subtarget_statistics_norm.index)))
# y = ccf_values[0]
# y_upper = ccf_values[1][:, 1] - y
# y_lower = y - ccf_values[1][:, 0]

# fig = go.Figure(
#     data=go.Scatter(x=x, y=y, error_y=dict(type="data", symmetric=False, array=y_upper, arrayminus=y_lower))
# )

# fig.update_layout(title="Rain Rate and Lidar Intensity Cross-correlation")
# fig.show()